In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

from metadPy.bayesian import hmetad
from metadPy.utils import discreteRatings

from itertools import repeat
import tqdm
import pingouin as pg
from systole.hrv import time_domain, frequency_domain, nonlinear_domain
from multiprocessing import Process, Manager, cpu_count, Pool
import arviz as az

sns.set_context('talk')
print("Number of cpu available: ", cpu_count())

Number of cpu available:  128


This notebook will create the summary dataframe containing the HRD results (both Psi and post-hoc estimates), the HRV metrics and the questionnaires.

# Imports

In [2]:
path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
subjects = [sub[:-3] for  sub in os.listdir(f"{path}/data/raw/")]
subjects = [sub for sub in subjects if sub.isdigit()]  # Drop invalid filenames

In [3]:
# Questionnaires before and after the haunted house
scales_df = pd.read_excel(f"{path}/data/scales.xlsx")
scales_df = scales_df.rename(columns={"Participant": "participant_id"})

/opt/anaconda3/lib/python3.8/site-packages/outdated/utils.py:14: OutdatedCacheFailedWarning: Failed to use cache while checking for outdated package.
Set the environment variable OUTDATED_RAISE_EXCEPTION=1 for a full traceback.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


# Summary

## Behaviors

In [4]:
hrd_df = pd.DataFrame({"participant_id": subjects})

hrd_df[["decision_rt_mean", "decision_rt_median", 
        "confidence_mean", "confidence_median", 
        "confidence_rt_mean", "confidence_rt_median",
        "psi_threshold", "psi_slope"]
      ] = "NaN"

In [5]:
for sub in hrd_df.participant_id.unique():

    # Load subject level behavior file
    try:
        behavior_df = pd.read_csv(f"{path}/data/raw/{sub}HRD/{sub}HRD_final.txt")
    except:
        print(f"Subject {sub}: file not found.")
        hrd_df = hrd_df[hrd_df.participant_id != sub]
        continue

    row_filter = (hrd_df.participant_id == sub) # Filter the group level df

    # Decision
    hrd_df.loc[row_filter, "decision_rt_mean"] = behavior_df.DecisionRT.dropna().mean()
    hrd_df.loc[row_filter, "decision_rt_median"] = behavior_df.DecisionRT.dropna().median()

    # Confidence rating
    hrd_df.loc[row_filter, "confidence_mean"] = behavior_df.Confidence.dropna().mean()
    hrd_df.loc[row_filter, "confidence_median"] = behavior_df.Confidence.dropna().median()
    hrd_df.loc[row_filter, "confidence_rt_mean"] = behavior_df.ConfidenceRT.dropna().mean()
    hrd_df.loc[row_filter, "confidence_rt_median"] = behavior_df.ConfidenceRT.dropna().median()

    # Psi estimates
    hrd_df.loc[row_filter, "psi_threshold"] = behavior_df.EstimatedThreshold.dropna().iloc[-1]
    hrd_df.loc[row_filter, "psi_slope"] = behavior_df.EstimatedSlope.dropna().iloc[-1]

In [6]:
hrd_df["participant_id"] = hrd_df["participant_id"].astype(int)
summary_df = pd.merge(scales_df, hrd_df, on=["participant_id"], how="outer")
summary_df.to_csv(f"{path}/data/summary.csv", sep="\t", index=False)

## Metacognition

In [7]:
metad_df = pd.DataFrame([])
for sub in subjects:
    try:
        metad_sumary = az.summary(az.from_netcdf(Path(path, "data", "metad", f"{sub}_meyad.nc")))
        
        metad_df = pd.concat(
            [
                metad_df,
                pd.DataFrame(
                {
                    "participant_id": int(sub),
                    "d": [metad_sumary["mean"]["d1"]],
                    "c": [metad_sumary["mean"]["c1"]],
                    "meta_d": [
                        metad_sumary["mean"]["meta_d"]
                    ],
                    "m_ratio": [
                        metad_sumary["mean"]["meta_d"] / metad_sumary["mean"]["d1"]
                    ],
                }
            )
            ]
        )
    except FileNotFoundError:
        print(f"No metad for participant {sub}.")

No metad for participant 236.
No metad for participant 110.
No metad for participant 244.
No metad for participant 190.
No metad for participant 198.
No metad for participant 85.


Save the outputs from Bayesian modelling in the main dataframe.

In [8]:
summary_df = pd.merge(metad_df, summary_df, on=["participant_id"], how="outer")

In [9]:
print(f"{summary_df['meta_d'].isnull().sum()} participants with invalid meta-d.")

11 participants with invalid meta-d.


In [10]:
summary_df.to_csv(f"{path}/data/summary.csv", sep="\t", index=False)

# Heart rate variability

In [11]:
raw_df = pd.read_csv("/home/nicolas/git/HauntedHearts/data/raw/summary.csv", sep=",")

In [12]:
raw_df.Enter_time = pd.to_datetime(raw_df.Date + " " +  raw_df.Enter_time)
raw_df.Exit_time = pd.to_datetime(raw_df.Date + " " +  raw_df.Exit_time)

In [13]:
raw_df["TaskDuration"] = raw_df.Exit_time - raw_df.Enter_time 
raw_df["TaskDuration"] = raw_df["TaskDuration"].dt.total_seconds()

In [14]:
hrv_df = pd.DataFrame([])
for sub in subjects:
    
    if not sub.isdigit():
        continue

    # Extract RR time serie
    try:
        rr = pd.read_csv(
            f"{path}/data/heartrate/" + [file for file in os.listdir(f"{path}/data/heartrate/") if file.startswith(f"P{sub} ")][0],
            skiprows=[2], header=3, sep=";"
        ).ArtifactCorrectedRRVector.to_numpy()

        artifacts = pd.read_csv(
            f"{path}/data/heartrate/" + [file for file in os.listdir(f"{path}/data/heartrate/") if file.startswith(f"P{sub} ")][0],
            skiprows=[2], header=3, sep=";"
        ).RawArtifactVector.to_numpy()

        # When the recording started
        start_recording = pd.to_datetime(
            pd.read_csv(
                f"{path}/data/heartrate/" + [file for file in os.listdir(f"{path}/data/heartrate/") if file.startswith(f"P{sub} ")][0],
                sep=";", nrows=1, 
            ).loc[0][0][12:],
            dayfirst=True
        )

    except IndexError:
        print(f"No data for {sub}")
        
    if len(raw_df[raw_df.participant_id == int(sub)]) == 0:
        continue

    start_delay = (raw_df[raw_df.participant_id == int(sub)].Enter_time - start_recording).dt.total_seconds().values[0]

    duration = raw_df[raw_df.participant_id == int(sub)].TaskDuration.values[0]

    # Remove pre-recording time
    rr = rr[(np.cumsum(rr) / 1000) > start_delay]

    # Remove post-recording time
    rr = rr[(np.cumsum(rr) / 1000) < duration]
    
    if np.any(rr > 10000):
        continue
    
    if len(rr) > 0:
        this_df = pd.concat([
            time_domain(rr=rr, input_type="rr_ms"),
            frequency_domain(rr=rr, input_type="rr_ms"),
            nonlinear_domain(rr=rr, input_type="rr_ms")
        ])

        this_df = pd.pivot_table(this_df, values='Values', columns='Metric')
        this_df["participant_id"] = int(sub)
        this_df["per_artefacts"] = round((sum(artifacts) / len(artifacts)) * 100, 2)

        hrv_df = pd.concat([hrv_df, this_df])

    else:
        print(f"No RR time series for {sub}")

No RR time series for 246
No RR time series for 247
No RR time series for 76
No RR time series for 70
No RR time series for 248
No RR time series for 110
No RR time series for 213
No data for 31
No data for 42
No data for 149
No RR time series for 149
No RR time series for 75
No RR time series for 214
No data for 19
No RR time series for 69
No RR time series for 108
No RR time series for 249
No RR time series for 109


In [15]:
# Merge the HRV results with the main dataframe
summary_df = pd.merge(hrv_df, summary_df, on=["participant_id"], how="outer")

In [16]:
summary_df.to_csv(f"{path}/data/summary.csv", sep="\t", index=False)

In [17]:
%load_ext watermark
%watermark -n -u -v -iv -w -p metadPy,pymc

Last updated: Fri Oct 21 2022

Python implementation: CPython
Python version       : 3.8.8
IPython version      : 8.4.0

metadPy: 0.0.1
pymc   : 4.2.2

seaborn   : 0.12.1
arviz     : 0.12.1
matplotlib: 3.4.3
numpy     : 1.19.0
pingouin  : 0.5.2
pandas    : 1.4.1
tqdm      : 4.62.3

Watermark: 2.3.1

